# 2.0 Preprocessing 

Combines what was previously 2.0 and 2.1 into a unified notebook to run 

Output is in ```../working-vars/regression/inputs/<pc_params>```


2.1_process_pco2_data 
- Start with clustered, QC'ed open ocean data from section 1. 
Convert all data to pCO2 (choose which correction to use for floats, convert fco2 to pco2 for socat)
- Add MLD to coreDS
- Add seasonal time
- Save as regression inputs 


***Note that delta-pco2 is OCEAN - ATMOSPHERE***

In [1]:
import mod_loading as loader
from importlib import reload
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import gsw
import mod_argo
import matplotlib.pyplot as plt
from cmocean import cm as cmo


# === CUSTOM FUNCTIONS
import mod_preprocessing as mod_prep
import mod_ocean


# Import data

In [2]:
# reload(loader)
# [bgcDS, bgcINDEX]= loader.import_bgc_data()

In [3]:
# Results of GMM, use bgcArgo and SOCAT data for supervised regression training
reload(loader)

# ==== Choose which PC/GMM results to load
pcm_params = 'pc8_gmm7'

# ==== Use loader module to retrieve results from section 1
[bgcINDEX, bgcDS, socatDS] = loader.import_clustered_data(pcm_params=pcm_params)
# [coreDS, coreINDEX] = loader.import_core_data(type='L3_only')
# [coreDS, coreINDEX] = loader.import_core_data(type='processed')

# Marine boundary layer atmospheric CO2 data
# mbl_co2 = pd.read_csv('../data/marineboundarylayer/co2_mbl_2014-2018_data_tabular.csv', index_col=0)
# mbl_co2 = xr.open_dataset('../data/marineboundarylayer/co2_mbl_2014-2023_dataset_combined.nc')

In [ ]:
[bgcDS_test, bgcINDEX_test] = loader.import_bgc_data(type='test')
bgcINDEX_test, bgcDS_test = mod_prep.add_mixedlayer_pressure(bgcINDEX_test, bgcDS_test)

In [32]:
[coreDS, coreINDEX] = loader.import_core_data(type='processed') # has adt, sla, mld

OSError: [Errno -101] NetCDF: HDF error: '/Volumes/crusoe-repo/data/core/L3-interp/coreINDEX_valid_interp_2014-2023_acc20250424.nc'

In [ ]:
# Add MLD
reload(mod_prep)
bgcINDEX, bgcDS = mod_prep.add_mixedlayer_pressure(bgcINDEX, bgcDS)
# coreINDEX, coreDS = mod_prep.add_mixedlayer_pressure(coreINDEX, coreDS)

# To access the output in another notebook, run:
# [bgcDS, bgcINDEX] = loader.import_bgc_data(L3_only=False)
# datetag = '20260121'
# coreINDEX = xr.open_dataset('../working-vars/argo/import-L3/coreINDEX_clustered_pc8_gmm6_2014-2023_withMLD_acc' + datetag + '.nc')
# coreDS = xr.open_dataset('../working-vars/argo/import-L3/coreDATA_clustered_pc8_gmm6_2014-2023_withMLD_acc' + datetag + '.nc')


/opt/homebrew/Caskroom/mambaforge/base/envs/cremas/lib/python3.9/site-packages/xarray/coding/times.py:254: RuntimeWarning: invalid value encountered in cast
  flat_num_dates_ns_int = (flat_num_dates * _NS_PER_TIME_DELTA[delta]).astype(


In [55]:
pcm_params

'pc8_gmm6'

# Process SOCAT fCO2 to pCO2

In [ ]:
# If rerunning, this function gives back the shipDF with processed fco2
[_, shipDF] = loader.import_processed_inputs()

In [4]:
[coreDS, coreINDEX] = loader.import_core_data(type = 'processed')

In [ ]:
# ==== Convert SOCAT fugacity to partial pressure pCO2
# Updated Feb 06 2026 # Jan 21 2026

shipDF = mod_prep.convert_socat_fco2(socatDS)
shipDF['mld'] = shipDF.nearest_profid.apply(lambda x: coreINDEX.sel(profid=x).mld.values)

shipDF = mod_prep.add_regression_time_vars(shipDF)
shipDF = mod_prep.add_regression_carbon_vars(shipDF, convert_pco2=True) 
# shipDF = mod_prep.convert_co2_ppm_to_pco2(shipDF)
shipDF['delta_pco2'] = shipDF['pco2_ocean'] - shipDF['pco2_atmos']

# result = socatDF[['fco2rec', 'pco2']].head(100)
# result['percent_diff'] = result.apply(lambda row: (row.pco2 - row.fco2rec)/row.fco2rec * 100, axis=1)
# result.describe()

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [11]:
result = shipDF_processed[['atmos_co2_ppm', 'pco2_atmos']].head(1000)
result['percent_diff'] = result.apply(lambda row: (row.pco2_atmos - row.atmos_co2_ppm)/row.atmos_co2_ppm * 100, axis=1)
result.describe()

,pco2_atmos,percent_diff
count,1000.000000,1000.000000
mean,398.006527,-1.667455
std,6.022208,1.184274
min,376.632009,-5.207693
25%,394.151272,-2.460089
50%,398.526932,-1.624621
75%,402.276366,-0.774142
max,413.089665,1.608417


In [ ]:
use_cols = [ 'cruiseID', 'expoID', 'sid', 'cluster', 'datetime', 
    'longitude', 'latitude', 'linear_time',
    'nearest_profid', 'prof_datetime', 'prof_lat', 'prof_lon', 'yd_sep', 'km_sep', 
    'ydcos', 'ydsin',
    'pressure', 'CT', 'SA', 'mld',  'sst', 'sss',
    'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
    'fco2rec', 
    'pco2_atmos', 'pco2_ocean', 'delta_pco2']


shipDF_processed = shipDF.reset_index()[use_cols].copy()


In [11]:
# ==== Optional Save  # better to save after adding adt
# save=True
# datetag = datetime.now().strftime('%Y%m%d')

# # Choose final variables
# final_ship = shipDF_processed.rename(columns={'expoID': 'sampleid', 'cruiseID':'cruiseid'})
# save_cols = ['sampleid', 'cruiseid', 'cluster', 'nearest_profid',
#         'datetime', 'latitude', 'longitude', 
#         'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
#         'CT', 'SA', 'mld', 'atm_pres_hPa',
#         'fco2rec', 'pco2', 
#         'pco2_atm', 'delta_pco2']
# shipDF_final = final_ship[save_cols]

# if save:
#     savepath = '../working-vars/regression/inputs/'
#     shipDF_final.to_csv(savepath + 'shipDF_socatv2024_1d_pCO2converted_co2sys_PCM_8pc8class_acc' + datetag + '.csv')

#     print('Saved ship data to ' + savepath + ', datetag ' + datetag)

Saved ship data to ../working-vars/regression/inputs/, datetag 20260121


# Process SOCCOM float corrected pCO2 (surface 20m averages)

In [ ]:
# ==== Process corrected pH to pCO2 data from 20m averages
# Updated Feb 06 2026
use_var = 'pCO2_pHbias5_pK1'
print('Using corrected variable: <' + use_var + '> for float pCO2')
soccom_df = mod_prep.process_soccom_20m_clusters(bgcDS, use_var=use_var) # using new clustered bgcDS
soccom_df['pco2_ocean'] = soccom_df[use_var]
# floatDF['pco2'] = soccom_df.reset_index()[use_var]
# floatDF_processed = mod_prep.add_regression_vars(floatDF)

Using corrected variable: <pCO2_pHbias5_pK1> for float pCO2
Clusters assigned.
Initially missing clusters for 623 profiles, filling in ==>
Filled in 296 missing clusters using adjacent profiles


In [ ]:
soccom_df = mod_prep.add_regression_time_vars(soccom_df)
soccom_df = mod_prep.add_regression_carbon_vars(soccom_df, convert_pco2=True) 
# soccom_df = mod_prep.convert_co2_ppm_to_pco2(soccom_df)
soccom_df['delta_pco2'] = soccom_df['pco2_ocean'] - soccom_df['pco2_atmos']

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [ ]:
use_cols = ['wmoid', 'profid', 'prof_datetag', 'cluster', 'datetime', 
        'latitude', 'longitude', 'ydcos', 'ydsin',
        'CT', 'SA', 'mld',  'sst', 'sss',
        'atmos_pres_Pa', 'atmos_pres_atm',  'vapor_pres_atm',
        'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
        'pco2_atmos', 'pco2_ocean', 'delta_pco2']

floatDF_processed = soccom_df.reset_index()
floatDF_processed = floatDF_processed[use_cols].copy()

In [13]:
# === OPTIONAL SAVE # better to save after adding adt
# save = True
# savepath = '../working-vars/regression/inputs/'
# float_parameter = 'pHbias5_pK1'
# datetag = datetime.now().strftime('%Y%m%d')

# # Choose final variables
# save_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
#             'datetime', 'latitude', 'longitude', 
#             'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
#             'CT', 'SA', 'mld', 
#             'pCO2_standard', 'pCO2_pHbias3',
#             'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1' ,'pco2', 
#             'pco2_atm', 'delta_pco2']
# floatDF_final = floatDF_processed[save_cols]

# if save:
#     floatDF_final.to_csv(savepath + 'floatDF_soccom20m_pco2_' + float_parameter + '_PCM_8pc8class_acc' + datetag + '.csv')
#     print('Saved float data to ' + savepath + ', datetag ' + datetag)

Saved float data to ../working-vars/regression/inputs/, datetag 20260121


# Add SSH and wind products as features 


Global Ocean Gridded L 4 Sea Surface Heights And Derived Variables Reprocessed 1993 Ongoing

https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L4_MY_008_047/description



- Variable abbrevations (https://www.cmcc.it/wp-content/uploads/2021/03/CMEMS-MED-PUM-006-013-V1.1.pdf)

In [34]:
# Use save columns
# floatDF = floatDF_final.copy()
# shipDF = shipDF_final.copy()

# Import daily satellite ADT at 1/4 deg resolution
adt_dict = {k:None for k in range(2014,2024)}
for open_yr in adt_dict.keys():
    folder = '/Volumes/crusoe-repo/data/copernicusmarine/' # used to be in OneDrive/Code/CRUSOE
    filepath = ('cmems_obs-sl_glo_phy-ssh_my_allsat-l4-duacs-0.125deg_P1D_adt-sla_179.94W-179.94E_88.94S-35.06S_' + str(open_yr) + '-01-01-' + str(open_yr) + '-12-31.nc')
    adt_dict[open_yr] = xr.open_dataset(folder + filepath)



In [38]:
floatDF_processed

,wmoid,profid,prof_datetag,cluster,datetime,latitude,longitude,ydcos,ydsin,CT,...,atmos_pres_atm,vapor_pres_atm,pCO2_standard,pCO2_pHbias3,pCO2_pHbias5,pCO2_pHbias3_pK1,pCO2_pHbias5_pK1,pco2_atmos,pco2_ocean,delta_pco2
0,2903456,2903456_id001,2903456_20230609,6.0,2023-06-09 13:55:00.000001,-55.053,200.071,-0.920775,0.390093,6.091327,...,0.965507,0.009102,429.720062,426.419117,424.231433,417.619562,415.480797,398.006232,415.480797,17.474565
1,2903456,2903456_id002,2903456_20230619,6.0,2023-06-19 20:54:00.000001,-55.032,200.389,-0.975083,0.221842,5.708396,...,0.984949,0.008864,433.260472,429.939403,427.738371,421.047626,418.895846,406.265412,418.895846,12.630434
2,2903456,2903456_id003,2903456_20230630,6.0,2023-06-30 03:51:59.999995,-55.004,200.812,-0.998910,0.046669,5.783078,...,0.968270,0.008909,433.606383,430.282026,428.078816,421.384905,419.230993,399.453450,419.230993,19.777543
3,2903456,2903456_id004,2903456_20230710,6.0,2023-07-10 10:54:00.000005,-54.900,201.404,-0.991513,-0.130011,5.685898,...,0.984752,0.008850,426.079908,422.809990,420.642863,414.076405,411.957753,406.435285,411.957753,5.522468
4,2903456,2903456_id005,2903456_20230720,6.0,2023-07-20 18:19:00.000002,-54.722,202.200,-0.953027,-0.302887,5.851405,...,0.970442,0.008952,427.685797,424.402500,422.226509,415.639035,413.511711,400.541767,413.511711,12.969945
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
11172,5906583,NaN,5906583_20231110,4.0,2023-11-10 05:56:00.000004,-62.603,163.310,0.622491,-0.782627,-1.736849,...,0.989983,0.005205,436.629351,433.365990,431.203083,424.171180,422.057030,411.476586,422.057030,10.580444
11173,5906583,5906583_id031,5906583_20231120,4.0,2023-11-20 12:59:00.000000,-62.689,162.887,0.750620,-0.660735,-1.672186,...,1.001431,0.005230,422.934803,419.767349,417.668016,410.878138,408.826105,416.243436,408.826105,-7.417331
11174,5906583,5906583_id032,5906583_20231130,4.0,2023-11-30 19:55:59.999998,-62.720,162.534,0.855236,-0.518240,-1.683684,...,0.988305,0.005226,415.820134,412.703188,410.637335,403.970936,401.951619,410.705222,401.951619,-8.753604
11175,5906583,5906583_id033,5906583_20231211,4.0,2023-12-11 04:32:00.000003,-62.700,162.647,0.933550,-0.358446,-1.182184,...,0.981100,0.005423,412.723012,409.624079,407.570172,400.971131,398.963470,407.573351,398.963470,-8.609881


In [39]:
shipDF_processed

,cruiseID,expoID,sid,cluster,datetime,longitude,latitude,datetime,linear_time,nearest_profid,...,mld,sst,sss,atmos_pres_Pa,atmos_pres_atm,vapor_pres_atm,fco2rec,pco2_atmos,pco2_ocean,delta_pco2
0,09AR20140129,09AR20140129_id0,46,1,2014-01-31 11:31:33,140.34900,-48.276570,2014-01-31 11:31:33,30.480243,5903811_id084,...,15.314093,11.4590,34.852908,99660.0,0.983568,0.013091,345.0500,382.002560,346.358569,-35.643991
1,09AR20140309,09AR20140309_id0,58,1,2014-03-10 12:19:07,144.05060,-47.839280,2014-03-10 12:19:07,68.513275,5903226_id160,...,55.648975,11.2950,34.444003,101620.0,1.002911,0.012952,361.2390,389.676693,362.611869,-27.064824
2,09AR20140309,09AR20140309_id3,83,1,2014-04-17 11:44:53,134.95110,-47.322995,2014-04-17 11:44:53,106.489502,1901436_id156,...,125.091019,11.1540,34.776184,100980.0,0.996595,0.012829,361.2860,387.744652,362.661553,-25.083100
3,09AR20141022,09AR20141022_id0,85,1,2014-10-22 18:51:25,145.86920,-44.039315,2014-10-22 18:51:25,294.785706,5903673_id132,...,125.709048,12.6550,35.541646,100150.0,0.988404,0.014159,359.6375,385.842431,360.980542,-24.861890
4,09AR20141022,09AR20141022_id0,86,1,2014-10-23 11:34:07,142.29540,-45.169150,2014-10-23 11:34:07,295.482025,5903811_id110,...,304.263107,11.3910,35.301926,99810.0,0.985048,0.013029,368.4390,384.972942,369.837497,-15.135446
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
6082,PAT520230420,PAT520230420_id0,6279,7,2023-04-20 18:26:00,172.14230,-39.021790,2023-04-20 18:26:00,3396.768056,5905509_id054,...,65.539056,19.2070,35.629932,100230.0,0.989193,0.021527,382.5310,402.074370,383.845749,-18.228621
6083,PAT520230528,PAT520230528_id3,6281,7,2023-06-18 10:06:29,168.18820,-37.263285,2023-06-18 10:06:29,3455.421169,5905507_id046,...,115.486492,17.0670,35.815094,102550.0,1.012090,0.018816,363.3050,413.572284,364.587659,-48.984625
6084,PAT520230620,PAT520230620_id0,6282,7,2023-06-20 17:15:00,172.78090,-38.389500,2023-06-20 17:15:00,3457.718750,5906635_id093,...,63.819400,16.7790,35.646088,100530.0,0.992154,0.018477,364.9090,405.379731,366.202011,-39.177720
6085,PAT520231011,PAT520231011_id1,6286,7,2023-10-15 15:56:30,177.57530,-37.193580,2023-10-15 15:56:30,3574.664236,5906421_id087,...,105.351299,15.6535,35.478500,102440.0,1.011004,0.017199,350.2155,415.096687,351.474244,-63.622443


In [40]:
# Run for all ten years
allplat =  pd.concat([floatDF_processed, shipDF_processed], axis=0, ignore_index=True)
allplat =  mod_ocean.expand_datetime(allplat, type='dataframe')
allplat_annual = {k:None for k in range(2014,2024)} # store by year

def get_adt_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').adt.values.tolist()
def get_sla_for_obs(row):
    return adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest').sla.values.tolist()

for yr in range(2014,2024):
    allplat_annual[yr] = allplat[allplat.year==yr]
    adt_data = adt_dict[yr]
    #  allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest'), axis=1)
    allplat_annual[yr]['adt'] =  allplat_annual[yr].apply(lambda row: get_adt_for_obs(row), axis=1)
    allplat_annual[yr]['sla'] =  allplat_annual[yr].apply(lambda row: get_sla_for_obs(row), axis=1)

# Separate into ship and float
allplat_added = pd.concat(allplat_annual.values(), axis=0)
shipDF_added = allplat_added[allplat_added.wmoid.isna()]

floatDF_added = allplat_added[~allplat_added.wmoid.isna()]
floatDF_added['wmoid'] = floatDF_added['wmoid'].astype(int)

InvalidIndexError: Reindexing only valid with uniquely valued Index objects

In [31]:
# === Optional save: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/' + pcm_params + '/'

    # Floats
    filename = 'floatDF_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_' + pcm_params + '_acc' + datetag + '.csv'
    use_cols = ['profid', 'wmoid', 'prof_datetag', 'cluster', 
              'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pCO2_standard', 'pCO2_pHbias3', 'pCO2_pHbias5', 'pCO2_pHbias3_pK1', 'pCO2_pHbias5_pK1', 
              'pco2', 'pco2_atm', 'delta_pco2',
              'year', 'month', 'day', 
              'adt', 'sla']
    floatDF_added[use_cols].to_csv(filepath + filename)
    print('Saved floatDF to ' + filepath + filename)

    # Ship data
    filename = 'shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_' + pcm_params + '_acc' + datetag + '.csv'
    use_cols = ['sampleid', 'cruiseid', 'nearest_profid', 
              'cluster', 'datetime', 'latitude', 'longitude', 
              'linear_time', 'ydcos', 'ydsin', 'decimalyr', 
              'CT', 'SA', 'mld', 
              'pco2', 'pco2_atm', 'delta_pco2', 
              'atm_pres_hPa', 'fco2rec', 
              'year', 'month', 'day', 
              'adt','sla']
    shipDF_added[use_cols].to_csv(filepath + filename)
    print('Saved shipDF to ' + filepath + filename)
    print(datetime.now().strftime('%Y-%m-%d %H:%M:%S'))



Saved floatDF to ../working-vars/regression/inputs/pc8_gmm7/floatDF_ADT_SLA_soccom20m_pCO2_pHbias5_pK1_PCM_pc8_gmm7_acc20260204.csv
Saved shipDF to ../working-vars/regression/inputs/pc8_gmm7/shipDF_ADT_SLA_socatv2024_1d_pCO2converted_co2sys_PCM_pc8_gmm7_acc20260204.csv
2026-02-04 14:25:23


# Add ADT/SLA to Core Argo (LONG RUN TIME)


- Warning: long run time, ~18 hours
- Run already saved in ```../working-vars/regression/inputs/core/coreDS_yrs2014-2023_ADT_SLA_processed_acc20260201.nc```

In [11]:
# coreDF = coreDS.to_dataframe().reset_index().copy()

# coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
# coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
# coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
# coreDF = mod_ocean.add_decimalyr(coreDF)


# reload(mod_prep)
# mod_prep.add_regression_vars(coreDF, timeOnly=True)

/Users/sangminsong/Library/CloudStorage/OneDrive-UW/Code/CRUSOE/src/mod_ocean.py:34: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  (df['datetime'].dt.is_leap_year.replace({True: 366, False: 365}))


In [ ]:
coreDF = coreDS.to_dataframe().reset_index().copy()

coreDF['linear_time'] = coreDF.datetime.apply(lambda x: mod_ocean.datetime2ytd(np.datetime64(x), ref_time='2014-01-01'))
coreDF['ydcos']= mod_ocean.get_ydsines(coreDF.linear_time.values)[0]
coreDF['ydsin']= mod_ocean.get_ydsines(coreDF.linear_time.values)[1]
coreDF = mod_ocean.add_decimalyr(coreDF)

coreDF =  mod_ocean.expand_datetime(coreDF, type='dataframe')



In [34]:
from tqdm import tqdm

In [35]:
# ==== COMMENT OUT TO KEEP OLD PROGRESS
# coreDF_annual = {k:None for k in range(2014,2024)} # store by year


# === Optional save: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 
filepath = '../working-vars/regression/inputs/core/'


# Run over all ten years
for yr in tqdm(range(2015,2024)):
    print('==> Processing year ' + str(yr))
    adt_data = adt_dict[yr]
    core_data = coreDF[coreDF.year==yr].copy()
    core_data['adt'] = np.tile(np.nan, len(core_data))
    core_data['sla'] = np.tile(np.nan, len(core_data))

    for ind, row in core_data.iterrows():
        closest_row = adt_data.sel(time=row.datetime, latitude=row.latitude, longitude=row.longitude, method='nearest')
        adtval = closest_row.adt.values.tolist()
        slaval = closest_row.sla.values.tolist()
        core_data.loc[ind, 'adt'] = adtval
        core_data.loc[ind, 'sla'] = slaval
   
    coreDF_annual[yr] = core_data

    if save:
        filename = 'coreDF_yr' + str(yr)+ '_ADT_SLA_acc' + datetag + '.csv'
        coreDF_annual[yr].to_csv(filepath + filename)
        print('Saved year ' + str(yr) + ' to ' + filename)



coreDF_added = pd.concat(coreDF_annual.values(), axis=0)

  0%|          | 0/9 [00:00<?, ?it/s]

==> Processing year 2015


 11%|█         | 1/9 [1:26:06<11:28:51, 5166.46s/it]

Saved year 2015 to coreDF_yr2015_ADT_SLA_acc20260130.csv
==> Processing year 2016


 22%|██▏       | 2/9 [2:58:19<10:27:53, 5381.94s/it]

...

Saved year 2021 to coreDF_yr2021_ADT_SLA_acc20260130.csv
==> Processing year 2022


 89%|████████▉ | 8/9 [11:36:27<1:23:26, 5006.55s/it]

Saved year 2022 to coreDF_yr2022_ADT_SLA_acc20260130.csv
==> Processing year 2023


100%|██████████| 9/9 [13:07:21<00:00, 5249.09s/it]  

Saved year 2023 to coreDF_yr2023_ADT_SLA_acc20260130.csv


In [37]:
# === Optional save of full DF : 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/core/'
    # Floats
    filename = 'coreDF_yrs2014-2023_ADT_SLA_processed_acc' + datetag + '.csv'
    coreDF_added.to_csv(filepath + filename)
    print('Saved coreDF to ' + filepath + filename)

Saved coreDF to ../working-vars/regression/inputs/core/coreDF_yrs2014-2023_ADT_SLA_processed_acc20260131.csv


In [38]:
coreDF_added.columns

Index(['profid', 'pressure', 'CT', 'SA', 'sigma0', 'spice', 'temperature',
       'salinity', 'yearday', 'latitude', 'longitude', 'wmoid', 'datetime',
       'mld', 'mlp', 'linear_time', 'ydcos', 'ydsin', 'decimalyr', 'year',
       'month', 'day', 'adt', 'sla'],
      dtype='object')

In [ ]:
reload(mod_prep)
coreDF_added

In [43]:
coreDS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    wmoid        (profid, pressure) float64 1.9e+06 1.9e+06 ... 7.901e+06
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables:
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    mld          (profid) float64 90.55 66.09 13.02 74.86 ... 82.53 71.18 19.52
    mlp          (profid) object 91.27 66.61 13.12 75.46 ... 83.29 71.83 19.69
Attributes:
    title:            Core float profiles with valid surface data (QC flags 1...
    pressure_levels:  Pressure levels in dbar: [0, 5, 10, 15, 20, 25, 30, 35,...
    source:           Argopy, expert mode; GDAC snapshot Jan 2025
    date:             20250424

In [ ]:
coreDS

In [39]:
coreDS_added = mod_argo.to_xr_dataset(coreDF_added)

In [46]:
temp = xr.Dataset.from_dataframe(coreDF_added.set_index(["profid", 'pressure']))
nc_attrs={'date':str(datetime.now()),
          'adt_type': "Global Ocean Gridded L 4 Sea Surface Heights And Derived Variables Reprocessed 1993 Ongoing",
          'adt_source': "https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L4_MY_008_047/description"}

argo_DS = temp.set_coords(['profid', 'datetime', 'yearday', 'latitude', 'longitude'])
argo_DS = argo_DS.assign_attrs(nc_attrs)
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...

In [51]:
# === Optional save of full xr Dataset: 
save = True
datetag = datetime.now().strftime('%Y%m%d') 

if save:
    filepath = '../working-vars/regression/inputs/core/'
    # Floats
    filename = 'coreDATA_processed_2014-2023_MLD_ADT_SLA_acc' + datetag + '.nc'
    argo_DS.to_netcdf(filepath + filename)
    print('Saved coreDS to ' + filepath + filename)

    # later accessed through loader.import_core_data(L3_only = False)

Saved coreDS to ../working-vars/regression/inputs/core/coreDS_yrs2014-2023_ADT_SLA_processed_acc20260201.nc


In [52]:
argo_DS

<xarray.Dataset>
Dimensions:      (profid: 328936, pressure: 197)
Coordinates:
  * profid       (profid) object '1900410_id260' ... '7901011_id005'
  * pressure     (pressure) int64 0 5 10 15 20 25 30 ... 955 960 965 970 975 980
    yearday      (profid, pressure) float64 1.107 1.107 ... 3.643e+03 3.643e+03
    latitude     (profid, pressure) float64 -40.36 -40.36 ... -51.79 -51.79
    longitude    (profid, pressure) float64 95.36 95.36 95.36 ... -46.82 -46.82
    datetime     (profid, pressure) datetime64[ns] 2014-01-02T02:34:12 ... 20...
Data variables: (12/18)
    CT           (profid, pressure) float64 12.71 12.71 12.7 ... 2.183 2.176
    SA           (profid, pressure) float64 35.03 35.03 35.03 ... 34.8 34.8 34.8
    sigma0       (profid, pressure) float64 26.35 26.35 26.35 ... 27.67 27.67
    spice        (profid, pressure) float64 1.69 1.69 1.688 ... -0.08716 -0.0865
    temperature  (profid, pressure) float64 12.72 12.72 12.71 ... 2.244 2.237
    salinity     (profid, pressure) float64 34.87 34.87 34.87 ... 34.63 34.63
    ...           ...
    decimalyr    (profid, pressure) float64 2.014e+03 2.014e+03 ... 2.024e+03
    year         (profid, pressure) int64 2014 2014 2014 2014 ... 2023 2023 2023
    month        (profid, pressure) int64 1 1 1 1 1 1 1 ... 12 12 12 12 12 12 12
    day          (profid, pressure) int64 2 2 2 2 2 2 2 ... 23 23 23 23 23 23 23
    adt          (profid, pressure) float64 0.5323 0.5323 ... -0.5156 -0.5156
    sla          (profid, pressure) float64 -0.012 -0.012 ... -0.0052 -0.0052
Attributes:
    date:        2026-02-01 12:29:20.321568
    adt_type:    Global Ocean Gridded L 4 Sea Surface Heights And Derived Var...
    adt_source:  https://data.marine.copernicus.eu/product/SEALEVEL_GLO_PHY_L...